# Agente 07 — Verificador y trazabilidad (productivo)
Notebook operacional delgado. La lógica científica permanece en `src/`.


## 1. Montaje y rutas

In [1]:
import os
import sys
import json
import tempfile
import importlib
from pathlib import Path

FIXTURE_MODE = os.environ.get("AGENT07_FIXTURE_MODE", "0") == "1"

REPO_ROOT = Path(
    os.environ.get(
        "AGENT07_REPO_ROOT",
        "/content/tesis_codigo" if not FIXTURE_MODE else Path.cwd(),
    )
).resolve()

PROJECT_ROOT = Path(
    os.environ.get(
        "AGENT07_PROJECT_ROOT",
        "/content/proyecto_estado_arte"
        if not FIXTURE_MODE
        else tempfile.mkdtemp(prefix="agent07_phase74_"),
    )
).resolve()

PROJECT_SRC = Path(
    os.environ.get(
        "AGENT07_PROJECT_SRC",
        str(PROJECT_ROOT / "src"),
    )
).resolve()

EXPERIMENT_ROOT = Path(
    os.environ.get(
        "AGENT07_EXPERIMENT_ROOT",
        str(PROJECT_ROOT / "experimento_paper_02"),
    )
).resolve()

# El repositorio contiene el paquete migrado `src`.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# 00_setup_config crea config.py y rag_policy.py en
# /content/proyecto_estado_arte/src. Esta ruta también debe estar disponible.
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

importlib.invalidate_caches()

if not FIXTURE_MODE:
    required_roots = {
        "repositorio": REPO_ROOT,
        "proyecto": PROJECT_ROOT,
        "configuración central": PROJECT_SRC,
        "experimento": EXPERIMENT_ROOT,
    }
    missing_roots = [
        f"{label}: {path}"
        for label, path in required_roots.items()
        if not path.exists()
    ]
    if missing_roots:
        raise FileNotFoundError(
            "Faltan rutas productivas requeridas por el Agente 07:\n- "
            + "\n- ".join(missing_roots)
        )

    required_config_modules = [
        PROJECT_SRC / "config.py",
        PROJECT_SRC / "rag_policy.py",
    ]
    missing_modules = [
        str(path) for path in required_config_modules if not path.is_file()
    ]
    if missing_modules:
        raise FileNotFoundError(
            "El notebook 00_setup_config no dejó los módulos requeridos:\n- "
            + "\n- ".join(missing_modules)
        )

print(
    {
        "fixture_mode": FIXTURE_MODE,
        "repo_root": str(REPO_ROOT),
        "project_root": str(PROJECT_ROOT),
        "project_src": str(PROJECT_SRC),
        "experiment_root": str(EXPERIMENT_ROOT),
    }
)


{'fixture_mode': False, 'repo_root': '/content/tesis_codigo', 'project_root': '/content/proyecto_estado_arte', 'project_src': '/content/proyecto_estado_arte/src', 'experiment_root': '/content/proyecto_estado_arte/experimento_paper_02'}


## 2. Imports del repositorio

In [2]:
from src.adapters.verification_notebook import (
    prepare_agent07_execution, execute_prepared_agent07, commit_executed_agent07,
    resume_agent07_execution, validate_agent07_artifact_manifest_contract,
    resolve_committed_agent06_output, AGENT07_ARTIFACT_NAMES,
)
from src.adapters.verification_runtime import Agent07RuntimeInput, build_agent07_runtime_dependencies
from src.adapters.agent06_verification_handoff import (
    Agent07RetrieverBinding,
    validate_agent07_experiment_compatibility,
    build_agent07_input_from_committed_agent06,
)
from src.state.state_store import StateStore
from importlib import import_module


## 3. Configuración explícita del Agente 07

In [3]:
from pathlib import Path

project_src = Path("/content/proyecto_estado_arte/src")

for name in ["config.py", "rag_policy.py"]:
    path = project_src / name
    print(
        "OK" if path.is_file() else "FALTA",
        path
    )

OK /content/proyecto_estado_arte/src/config.py
OK /content/proyecto_estado_arte/src/rag_policy.py


In [4]:
if FIXTURE_MODE:
    CANONICAL_STATE_PATH = (
        EXPERIMENT_ROOT
        / "05_outputs"
        / "00_orchestrator_planner"
        / "pipeline_state.json"
    )
    CANONICAL_OUTLINE_MAPPING_PATH = (
        EXPERIMENT_ROOT
        / "05_outputs"
        / "04_outline"
        / "outline_paper_mapping.csv"
    )
    AGENT07_OUTPUT_DIR = (
        EXPERIMENT_ROOT
        / "05_outputs"
        / "06_verification_traceability"
    )
    AGENT07_STAGING_DIR = (
        EXPERIMENT_ROOT
        / "05_outputs"
        / ".agent07_staging"
    )

    experiment_paths = {
        "code_root": str(REPO_ROOT),
        "project_root": str(PROJECT_ROOT),
        "experiment_root": str(EXPERIMENT_ROOT),
        "root": str(EXPERIMENT_ROOT),
        "pipeline_state_path": str(CANONICAL_STATE_PATH),
        "outline_paper_mapping_path": str(CANONICAL_OUTLINE_MAPPING_PATH),
        "agent07_output_dir": str(AGENT07_OUTPUT_DIR),
        "agent07_staging_dir": str(AGENT07_STAGING_DIR),
    }

    policy_versions = {"verification": "v1"}
    schema_versions = {
        "provisional_bundle": "v2",
        "multi_proposal_resolution": "v1",
        "runtime_metrics": "v1",
    }
    agent07_config = {
        "runtime_mode": "in_memory",
        "external_paper_search": False,
    }
    active_experiment_config = {}

else:
    try:
        from config import (
            EXPERIMENT_ID,
            EXPERIMENT_DIR,
            OUTPUTS_DIR,
            OUTLINE_DIR,
            ORCHESTRATOR_DIR,
            VERIFICATION_TRACEABILITY_DIR,
            CHUNKS_DIR,
            CHROMA_DIR,
            EMBEDDING_MODEL_NAME,
            OPENAI_MODEL,
            CHROMA_COLLECTION_NAME,
            VERIFICATION_POLICY,
            POST_CORRECTION_RECHECK_POLICY,
        )
        from rag_policy import get_rag_policy
    except (ModuleNotFoundError, ImportError) as exc:
        raise RuntimeError(
            "No se pudo cargar la configuración central creada por "
            f"00_setup_config desde {PROJECT_SRC}. Detalle: {exc}"
        ) from exc

    # No se vuelven a adivinar rutas. Se usan las constantes reales
    # definidas por /content/proyecto_estado_arte/src/config.py.
    CANONICAL_STATE_PATH = ORCHESTRATOR_DIR / "pipeline_state.json"
    CANONICAL_OUTLINE_MAPPING_PATH = (
        OUTLINE_DIR / "outline_paper_mapping.csv"
    )
    AGENT07_OUTPUT_DIR = VERIFICATION_TRACEABILITY_DIR
    AGENT07_STAGING_DIR = OUTPUTS_DIR / ".agent07_staging"

    CHROMA_MANIFEST_PATH = (
        CHROMA_DIR / "chroma_index_manifest.json"
    )
    CHUNKS_MANIFEST_PATH = (
        CHUNKS_DIR / "chunks_clean_for_rag.jsonl"
    )

    expected_paths = {
        "PipelineState": CANONICAL_STATE_PATH,
        "outline_paper_mapping.csv": CANONICAL_OUTLINE_MAPPING_PATH,
        "chroma_index_manifest.json": CHROMA_MANIFEST_PATH,
        "chunks_clean_for_rag.jsonl": CHUNKS_MANIFEST_PATH,
    }
    missing = [
        f"{name}: {path}"
        for name, path in expected_paths.items()
        if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Faltan entradas reales requeridas por el Agente 07:\n- "
            + "\n- ".join(missing)
        )

    experiment_paths = {
        "code_root": str(REPO_ROOT),
        "project_root": str(PROJECT_ROOT),
        "experiment_root": str(EXPERIMENT_DIR),
        "root": str(EXPERIMENT_DIR),
        "pipeline_state_path": str(CANONICAL_STATE_PATH),
        "outline_paper_mapping_path": str(
            CANONICAL_OUTLINE_MAPPING_PATH
        ),
        "agent07_output_dir": str(AGENT07_OUTPUT_DIR),
        "agent07_staging_dir": str(AGENT07_STAGING_DIR),
        "chroma_dir": str(CHROMA_DIR),
        "chunks_dir": str(CHUNKS_DIR),
    }

    from copy import deepcopy
    from src.config.verification_policy_config import (
        get_verification_input_policy,
        PROVISIONAL_BUNDLE_FINGERPRINT_VERSION,
    )
    from src.tools.verification.resolution import (
        RESOLUTION_FP_VERSION,
    )
    from src.adapters.verification_runtime import (
        AGENT07_RUNTIME_METRICS_VERSION,
    )

    complete_policy = get_verification_input_policy()

    verification_policy = deepcopy(complete_policy)
    verification_policy.update(deepcopy(VERIFICATION_POLICY))

    correction_policy = deepcopy(complete_policy)
    correction_policy.update(deepcopy(VERIFICATION_POLICY))

    reverification_policy = deepcopy(complete_policy)
    reverification_policy.update(
        deepcopy(POST_CORRECTION_RECHECK_POLICY)
    )

    agent07_config = {
        "runtime_mode": "productive",
        "external_paper_search": False,
        "verification_policy": verification_policy,
        "correction_policy": correction_policy,
        "reverification_policy": reverification_policy,
        "verification_prompt_version": complete_policy[
            "verification_user_prompt_version"
        ],
        "correction_prompt_version": complete_policy[
            "correction_user_prompt_version"
        ],
        "reverification_prompt_version": complete_policy[
            "reverification_user_prompt_version"
        ],
        "verification_budgets": {
            "max_llm_attempts": int(
                complete_policy["max_llm_attempts_per_claim"]
            ),
            "max_format_repair_attempts": int(
                complete_policy["max_format_repair_attempts"]
            ),
            "max_additional_retrieval_requests": int(
                complete_policy[
                    "max_additional_retrieval_requests"
                ]
            ),
        },
        "correction_budgets": {
            "max_llm_attempts": int(
                complete_policy["max_correction_llm_attempts"]
            ),
            "max_format_repair_attempts": int(
                complete_policy[
                    "max_correction_format_repair_attempts"
                ]
            ),
            "max_proposals_per_claim": int(
                complete_policy[
                    "max_correction_proposals_per_claim"
                ]
            ),
        },
        "reverification_budgets": {
            "max_llm_attempts": int(
                complete_policy[
                    "max_reverification_llm_attempts"
                ]
            ),
            "max_format_repair_attempts": int(
                complete_policy[
                    "max_reverification_format_repair_attempts"
                ]
            ),
        },
        "verification_model": OPENAI_MODEL,
        "correction_model": OPENAI_MODEL,
        "reverification_model": OPENAI_MODEL,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "collection_name": CHROMA_COLLECTION_NAME,
        "chroma_collection_name": CHROMA_COLLECTION_NAME,
    }

    policy_versions = {
        "verification": agent07_config[
            "verification_prompt_version"
        ],
        "correction": agent07_config[
            "correction_prompt_version"
        ],
        "reverification": agent07_config[
            "reverification_prompt_version"
        ],
    }

    schema_versions = {
        "provisional_bundle": (
            PROVISIONAL_BUNDLE_FINGERPRINT_VERSION
        ),
        "multi_proposal_resolution": RESOLUTION_FP_VERSION,
        "runtime_metrics": AGENT07_RUNTIME_METRICS_VERSION,
    }

    active_experiment_config = {
        "active_experiment_id": EXPERIMENT_ID,
        "experiment_dir": str(EXPERIMENT_DIR),
        "embedding_model": EMBEDDING_MODEL_NAME,
        "openai_model": OPENAI_MODEL,
        "chroma_collection_name": CHROMA_COLLECTION_NAME,
        "verification_policy": agent07_config[
            "verification_policy"
        ],
        "verification_prompt_version": agent07_config[
            "verification_prompt_version"
        ],
        "verification_budgets": agent07_config[
            "verification_budgets"
        ],
        "rag_policy": get_rag_policy(),
    }

    validate_agent07_experiment_compatibility(
        active_config=active_experiment_config,
        agent07_config=agent07_config,
        experiment_paths=experiment_paths,
    )

    # Snapshot reproducible de la configuración usada por 07.
    config_path = (
        EXPERIMENT_DIR
        / "config"
        / "agent07_runtime_config.json"
    )
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_payload = {
        "agent07_config": agent07_config,
        "policy_versions": policy_versions,
        "schema_versions": schema_versions,
        "generated_from": {
            "source": (
                "active_experiment.json + "
                "verification_policy_config.py"
            ),
            "experiment_id": EXPERIMENT_ID,
        },
    }
    config_path.write_text(
        json.dumps(
            config_payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    print("Configuración Agente 07:", config_path)
    print("PipelineState:", CANONICAL_STATE_PATH)
    print("Outline mapping:", CANONICAL_OUTLINE_MAPPING_PATH)
    print("Salida oficial 07:", AGENT07_OUTPUT_DIR)


Configuración Agente 07: /content/proyecto_estado_arte/experimento_paper_02/config/agent07_runtime_config.json
PipelineState: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/00_orchestrator_planner/pipeline_state.json
Outline mapping: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/04_outline/outline_paper_mapping.csv
Salida oficial 07: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/06_verification_traceability


## 4. Localización de salida committed del Agente 06

In [5]:
state_path = CANONICAL_STATE_PATH
store = StateStore(state_path)

print("PipelineState usado por Agente 07:", state_path)

if FIXTURE_MODE:
    sys.path.insert(
        0,
        str(REPO_ROOT / "tests" / "verification"),
    )
    from test_phase72_runtime_notebook_closure import (
        runtime_input as _fixture_input,
        deps as _fixture_deps,
    )

    fixture = _fixture_input()
    committed_agent06_output = dict(
        fixture.committed_agent06_output
    )
    agent07_config = dict(fixture.agent07_config)
    policy_versions = dict(fixture.policy_versions)
    schema_versions = dict(fixture.schema_versions)

    if not state_path.exists():
        from src.state.pipeline_state import (
            PipelineIdentity,
            PipelineState,
        )
        store.initialize(
            PipelineState(
                identity=PipelineIdentity(
                    "fixture",
                    "fixture-run",
                    "2026-01-01T00:00:00+00:00",
                    "2026-01-01T00:00:00+00:00",
                    "v1",
                )
            )
        )
else:
    committed_agent06_output = (
        build_agent07_input_from_committed_agent06(
            store=store,
            stage_name=agent07_config.get(
                "agent06_stage_name",
                "06_agente_redactor",
            ),
            agent07_config=agent07_config,
            policy_versions=policy_versions,
            schema_versions=schema_versions,
            experiment_paths=experiment_paths,
            outline_paper_mapping_path=(
                CANONICAL_OUTLINE_MAPPING_PATH
            ),
        )
    )

    print(
        "COMMIT 06 resuelto:",
        committed_agent06_output["agent06_decision_id"],
    )
    print(
        "Claims recibidos:",
        len(
            committed_agent06_output[
                "claim_verification_contexts"
            ]
        ),
    )


PipelineState usado por Agente 07: /content/proyecto_estado_arte/experimento_paper_02/05_outputs/00_orchestrator_planner/pipeline_state.json
COMMIT 06 resuelto: 89e04f41-0d06-4815-8cb2-a0a1b7d97fdf
Claims recibidos: 25


## 5. Construcción de dependencias

In [6]:
if FIXTURE_MODE:
    dependencies = _fixture_deps("COMPLETED")

else:
    import hashlib
    from dataclasses import asdict

    from llm_utils import get_llm
    from rag_utils import load_chroma_collection

    class Agent07ChromaRetriever:
        """Retriever productivo restringido a fuentes autorizadas."""

        def __init__(
            self,
            *,
            collection,
            experiment_id,
            collection_name,
            embedding_model,
            chroma_manifest_fingerprint,
            chunks_manifest_fingerprint,
            top_k=8,
            fetch_k=35,
        ):
            self.collection = collection
            self.experiment_id = experiment_id
            self.collection_name = collection_name
            self.embedding_model = embedding_model
            self.chroma_manifest_fingerprint = (
                chroma_manifest_fingerprint
            )
            self.chunks_manifest_fingerprint = (
                chunks_manifest_fingerprint
            )
            self.top_k = int(top_k)
            self.fetch_k = int(fetch_k)

        def retrieve_more(self, request):
            claim_id = str(request["claim_id"])
            claim_context = request["claim_context"]
            claim_text = str(
                claim_context.get("claim_text")
                or claim_context.get("original_claim_text")
                or ""
            ).strip()

            allowed_sources = tuple(
                request["allowed_source_filenames"]
            )
            allowed_set = set(allowed_sources)

            if not claim_text:
                raise ValueError(
                    "AGENT07_RETRIEVER_CLAIM_TEXT_MISSING"
                )

            result = self.collection.query(
                query_texts=[claim_text],
                n_results=self.fetch_k,
            )

            documents = (result.get("documents") or [[]])[0]
            metadatas = (result.get("metadatas") or [[]])[0]
            distances = (result.get("distances") or [[]])[0]

            selected = []
            seen_pairs = set()

            for native_rank, (text, metadata, distance) in enumerate(
                zip(documents, metadatas, distances),
                start=1,
            ):
                metadata = metadata or {}
                source = str(
                    metadata.get("source_filename") or ""
                ).strip()
                chunk_id = str(
                    metadata.get("chunk_id") or ""
                ).strip()
                text = str(text or "").strip()

                if (
                    not source
                    or not chunk_id
                    or not text
                    or source not in allowed_set
                ):
                    continue

                pair = (source, chunk_id)
                if pair in seen_pairs:
                    continue
                seen_pairs.add(pair)

                score = 1.0 - float(distance)
                selected.append(
                    {
                        "source_filename": source,
                        "chunk_id": chunk_id,
                        "text": text,
                        "retrieval_sources": ("chroma",),
                        "query_ids": (claim_id,),
                        "all_native_ranks": (native_rank,),
                        "native_ranks_by_retriever": {
                            "chroma": native_rank
                        },
                        "native_scores_by_retriever": {
                            "chroma": score
                        },
                        "native_score_types_by_retriever": {
                            "chroma": "cosine_similarity"
                        },
                        "first_seen_round": 1,
                        "last_seen_round": 1,
                        "fused_rrf_score": (
                            1.0 / (60.0 + native_rank)
                        ),
                        "text_variants": (),
                        "contradiction_signals": (),
                    }
                )

                if len(selected) >= self.top_k:
                    break

            return {
                "selected_candidates": tuple(selected),
                "rounds_executed": 1,
                "total_candidates_seen": len(documents),
                "total_unique_candidates_seen": len(seen_pairs),
                "queries_executed_total": 1,
                "new_unique_pairs_seen": len(seen_pairs),
                "queries": (
                    {
                        "query_id": claim_id,
                        "query_text": claim_text,
                    },
                ),
                "discarded_candidates": (),
                "retrieval_trace": (
                    {
                        "claim_id": claim_id,
                        "allowed_sources": allowed_sources,
                        "selected_count": len(selected),
                    },
                ),
                "contradiction_signals": (),
                "technical_issue_codes": (),
                "technical_status": "COMPLETED",
                "stop_reason": (
                    "STRUCTURAL_COVERAGE_SATISFIED"
                    if selected
                    else "NO_NEW_EVIDENCE"
                ),
                "queries_remaining": 0,
                "total_unique_candidates_retained": len(selected),
                "new_unique_pairs_selected": len(selected),
                "structural_coverage_improved": bool(selected),
                "structural_coverage_improved_this_delta": (
                    bool(selected)
                ),
            }

    def _sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as file:
            for block in iter(
                lambda: file.read(1024 * 1024),
                b"",
            ):
                digest.update(block)
        return digest.hexdigest()

    chroma_manifest = json.loads(
        CHROMA_MANIFEST_PATH.read_text(encoding="utf-8")
    )

    if (
        chroma_manifest.get("experiment_id")
        != committed_agent06_output["experiment_id"]
    ):
        raise ValueError(
            "El manifest de Chroma pertenece a otro experimento."
        )

    collection = load_chroma_collection(
        CHROMA_DIR,
        collection_name=CHROMA_COLLECTION_NAME,
        model_name=EMBEDDING_MODEL_NAME,
    )

    retriever_binding = Agent07RetrieverBinding(
        experiment_id=committed_agent06_output[
            "experiment_id"
        ],
        collection_name=CHROMA_COLLECTION_NAME,
        embedding_model=EMBEDDING_MODEL_NAME,
        chroma_manifest_fingerprint=_sha256(
            CHROMA_MANIFEST_PATH
        ),
        chunks_manifest_fingerprint=_sha256(
            CHUNKS_MANIFEST_PATH
        ),
    )

    incremental_retriever = Agent07ChromaRetriever(
        collection=collection,
        **asdict(retriever_binding),
        top_k=int(
            get_rag_policy()
            .get("retrieval_profiles", {})
            .get("default", {})
            .get("top_k", 8)
        ),
        fetch_k=int(
            get_rag_policy()
            .get("retrieval_profiles", {})
            .get("default", {})
            .get("fetch_k", 35)
        ),
    )

    verification_llm = get_llm(
        model=agent07_config["verification_model"],
        temperature=0.0,
    )
    correction_llm = get_llm(
        model=agent07_config["correction_model"],
        temperature=0.0,
    )
    reverification_llm = get_llm(
        model=agent07_config["reverification_model"],
        temperature=0.0,
    )

    dependencies = build_agent07_runtime_dependencies(
        config=agent07_config,
        experiment_paths=experiment_paths,
        verification_llm=verification_llm,
        correction_llm=correction_llm,
        reverification_llm=reverification_llm,
        incremental_retriever=incremental_retriever,
        active_experiment_config=active_experiment_config,
        retriever_binding=retriever_binding,
        chroma_manifest_path=str(CHROMA_MANIFEST_PATH),
        chunks_manifest_path=str(CHUNKS_MANIFEST_PATH),
        committed_experiment_id=(
            committed_agent06_output["experiment_id"]
        ),
    )

    print(
        "Dependencias productivas del Agente 07 construidas."
    )
    print(
        "Retriever:",
        {
            "experiment_id": retriever_binding.experiment_id,
            "collection_name": (
                retriever_binding.collection_name
            ),
            "embedding_model": (
                retriever_binding.embedding_model
            ),
        },
    )


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dependencias productivas del Agente 07 construidas.
Retriever: {'experiment_id': 'experimento_paper_02', 'collection_name': 'reference_papers_chunks', 'embedding_model': 'all-MiniLM-L6-v2'}


## 6. RESUME

In [7]:
runtime_input = Agent07RuntimeInput(committed_agent06_output, agent07_config, policy_versions, schema_versions, experiment_paths)
resume_result = resume_agent07_execution(store=store, runtime_input=runtime_input)
print("RESUME:", resume_result.action)


RESUME: NO_COMMIT


## 7. PREPARE

In [8]:
prepared = None
if resume_result.action in {"NO_COMMIT", "REEXECUTE"}:
    prepared = prepare_agent07_execution(store=store, runtime_input=runtime_input)
    print("PREPARED", prepared.decision_id, "attempt", prepared.attempt_number)

PREPARED 14ceb427-3276-41f2-8695-2a7bb7bdedfe attempt 1


## 8. EXECUTE

In [9]:
executed = resume_result.executed
if prepared is not None:
    executed = execute_prepared_agent07(store=store, prepared=prepared, dependencies=dependencies)
    print("EXECUTED", executed.runtime_result.runtime_status)

EXECUTED BLOCKED


## 9. Inspección de resultados candidatos

In [10]:
if executed is not None:
    rr = executed.runtime_result
    print({
        "runtime_status": rr.runtime_status,
        "aggregation_status": None if rr.provisional_bundle is None else rr.provisional_bundle.get("aggregation_status"),
        "resolution_status": None if rr.multi_proposal_resolution_result is None else rr.multi_proposal_resolution_result.get("resolution_status"),
        "candidate_artifacts": sorted(executed.candidate_payloads),
    })

{'runtime_status': 'BLOCKED', 'aggregation_status': None, 'resolution_status': None, 'candidate_artifacts': ['agent07_operational_audit.json', 'agent07_runtime_report.json']}


## 10. COMMIT explícito

In [11]:
if executed is not None and executed.runtime_result.runtime_status in {"COMPLETED", "PARTIAL"}:
    committed_state = commit_executed_agent07(store=store, executed=executed)
    print("COMMITTED", len(committed_state.decision_log))
else:
    print("COMMIT omitido: no existe resultado científico completo.")

COMMIT omitido: no existe resultado científico completo.


## 11. Validación post-COMMIT

In [12]:
post_resume = resume_agent07_execution(store=store, runtime_input=runtime_input)
print("POST_COMMIT_RESUME", post_resume.action)
output_dir = Path(experiment_paths["agent07_output_dir"])
if post_resume.action == "COMMITTED":
    assert all((output_dir / name).is_file() for name in AGENT07_ARTIFACT_NAMES)
    manifest = json.loads((output_dir / AGENT07_ARTIFACT_NAMES[-1]).read_text(encoding="utf-8"))
    validate_agent07_artifact_manifest_contract(manifest, artifact_bytes={name:(output_dir/name).read_bytes() for name in AGENT07_ARTIFACT_NAMES[:3]})
    state = store.load()
    assert state.pending_execution is None
    assert executed.runtime_result.correction_applied is False
    assert executed.runtime_result.evaluation_ready_emitted is False

POST_COMMIT_RESUME EXECUTED_NOT_COMMITTED


## 12. Resumen de artefactos

In [13]:
if post_resume.action == "COMMITTED":
    bundle = executed.runtime_result.provisional_bundle
    resolution = executed.runtime_result.multi_proposal_resolution_result
    metrics = bundle.get("metrics", {})
    plans = resolution.get("claim_resolution_plans", [])
    print({
        "runtime_status": executed.runtime_result.runtime_status,
        "aggregation_status": bundle.get("aggregation_status"),
        "resolution_status": resolution.get("resolution_status"),
        "claims_verified": metrics.get("claims_verified"),
        "corrections_proposed": metrics.get("corrections_proposed"),
        "accepted_issue_resolution_rate": metrics.get("accepted_issue_resolution_rate"),
        "claims_eligible_for_07c": sum(1 for p in plans if p.get("eligible_for_07c")),
        "claims_manual_review": sum(1 for p in plans if p.get("manual_review_required")),
        "normalized_bundle_fingerprint": bundle.get("normalized_bundle_fingerprint"),
        "resolution_fingerprint": resolution.get("multi_proposal_resolution_fingerprint"),
        "artifact_directory": str(output_dir),
        "resume_result": post_resume.action,
    })